<a href="https://colab.research.google.com/github/justamy20/scikit-learn-Cookbook/blob/main/Chapter_03.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bab 3: Teknik Pengurangan Dimensi (Dimensionality Reduction)

<div class="alert alert-info">
<b>Ringkasan Bab:</b> Dataset di dunia nyata seringkali memiliki puluhan bahkan ribuan fitur (High Dimensionality). Bab ini membahas teknik untuk mereduksi jumlah dimensi tersebut menggunakan Principal Component Analysis (PCA), Linear Discriminant Analysis (LDA), dan teknik non-linear seperti Kernel PCA serta t-SNE.
</div>

## 1. Principal Component Analysis (PCA)
PCA adalah teknik *unsupervised* yang mencari arah (komponen utama) yang memiliki varians terbesar dalam data. Komponen utama pertama adalah arah di mana data paling menyebar.

Secara matematis, PCA mencari *eigenvectors* ($v$) dan *eigenvalues* ($\lambda$) dari matriks kovarians data ($C$):
$$C v = \lambda v$$
di mana $C = \frac{1}{n-1} X^T X$ (asumsi data $X$ sudah di-pusatkan ke nol).

<div class="alert alert-warning">
<b>Peringatan Sangat Penting:</b> PCA sangat sensitif terhadap skala data! Pastikan untuk selalu menggunakan <code>StandardScaler</code> sebelum mengaplikasikan PCA agar setiap fitur memiliki bobot awal yang sama.
</div>

In [1]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import load_breast_cancer
import numpy as np

# Load dataset kanker payudara (memiliki 30 fitur)
data = load_breast_cancer()
X, y = data.data, data.target

# 1. Penskalaan (Wajib untuk PCA)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 2. Inisialisasi PCA untuk mereduksi dari 30 fitur menjadi 2 fitur saja
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

print(f"Bentuk data asli: {X.shape} (30 Fitur)")
print(f"Bentuk data setelah PCA: {X_pca.shape} (2 Fitur)")

# Melihat seberapa besar informasi (varians) yang berhasil dipertahankan
explained_variance = np.sum(pca.explained_variance_ratio_) * 100
print(f"Total informasi yang dipertahankan dalam 2 fitur: {explained_variance:.2f}%")

Bentuk data asli: (569, 30) (30 Fitur)
Bentuk data setelah PCA: (569, 2) (2 Fitur)
Total informasi yang dipertahankan dalam 2 fitur: 63.24%


---
## 2. Linear Discriminant Analysis (LDA)
Berbeda dengan PCA yang *unsupervised* (tidak melihat label), **LDA adalah teknik *supervised***. LDA mencoba mencari sumbu yang tidak hanya memaksimalkan varians data, tetapi juga **memaksimalkan pemisahan antar kelas**.

LDA bekerja dengan memaksimalkan rasio antara varians antar-kelas (*Between-class variance*, $S_B$) dan varians dalam-kelas (*Within-class variance*, $S_W$):
$$J(w) = \frac{w^T S_B w}{w^T S_W w}$$

In [2]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

# Kita gunakan dataset yang sama (X_scaled dan y)
# Inisialisasi LDA (mereduksi dimensi dengan melihat label 'y')
lda = LinearDiscriminantAnalysis(n_components=1) # Binary classification max n_components = classes - 1
X_lda = lda.fit_transform(X_scaled, y)

print(f"Bentuk data setelah LDA: {X_lda.shape} (1 Fitur)")
print(f"Varians yang dijelaskan oleh LDA: {lda.explained_variance_ratio_[0]*100:.2f}%")

Bentuk data setelah LDA: (569, 1) (1 Fitur)
Varians yang dijelaskan oleh LDA: 100.00%


---
## 3. Kernel PCA untuk Data Non-Linear
PCA standar hanya bisa menangkap hubungan linear (garis lurus). Bagaimana jika data kita berbentuk lingkaran atau pola non-linear rumit lainnya? Kita menggunakan **Kernel PCA** dengan trik kernel (seperti *Radial Basis Function* / RBF) untuk memetakan data ke dimensi yang lebih tinggi agar bisa dipisahkan.

In [3]:
from sklearn.decomposition import KernelPCA
from sklearn.datasets import make_circles

# Membuat data sintetis berbentuk dua lingkaran konsentris (non-linear)
X_circles, y_circles = make_circles(n_samples=400, factor=0.3, noise=0.05, random_state=42)

# Mencoba PCA biasa (Linear) - Akan gagal memisahkan struktur sirkular
pca_linear = PCA(n_components=2)
X_pca_linear = pca_linear.fit_transform(X_circles)

# Menggunakan Kernel PCA (RBF Kernel)
kpca = KernelPCA(n_components=2, kernel='rbf', gamma=15)
X_kpca = kpca.fit_transform(X_circles)

print("Data bentuk lingkaran berhasil diproses oleh Kernel PCA.")
print("Bentuk matriks output Kernel PCA:", X_kpca.shape)

Data bentuk lingkaran berhasil diproses oleh Kernel PCA.
Bentuk matriks output Kernel PCA: (400, 2)


---
## 4. t-SNE (t-Distributed Stochastic Neighbor Embedding)
**t-SNE** adalah algoritma khusus yang dirancang murni untuk **visualisasi** data dimensi tinggi ke dalam dimensi 2D atau 3D. Algoritma ini sangat ahli dalam mempertahankan struktur lokal (memastikan titik yang dekat di dimensi tinggi tetap dekat di dimensi rendah).

<div class="alert alert-info">
<b>Catatan:</b> t-SNE membutuhkan komputasi yang sangat berat. Untuk dataset yang sangat besar, sangat disarankan untuk melakukan PCA terlebih dahulu (misal: kurangi ke 50 dimensi pakai PCA, baru gunakan t-SNE untuk memetakannya ke 2 dimensi).
</div>

In [4]:
from sklearn.manifold import TSNE

# Kita terapkan pada sebagian kecil data kanker payudara untuk visualisasi
tsne = TSNE(n_components=2, random_state=42, perplexity=30)

# Mengubah data dari 30 fitur menjadi 2 fitur menggunakan t-SNE
X_tsne = tsne.fit_transform(X_scaled)

print("Bentuk data asli (Scaled):", X_scaled.shape)
print("Bentuk data setelah t-SNE :", X_tsne.shape)
print("Data sekarang siap untuk di-plot ke dalam grafik 2D (Sumbu X dan Y).")

Bentuk data asli (Scaled): (569, 30)
Bentuk data setelah t-SNE : (569, 2)
Data sekarang siap untuk di-plot ke dalam grafik 2D (Sumbu X dan Y).
